In [7]:
import time
import numpy as np
import sys
import pandas as pd
# import nrm
import csv
import subprocess
import os
import tarfile
import random
from datetime import datetime
import torch

In [8]:
import torch.nn.functional as F

def normalize(data, MIN, MAX):
    return np.round((np.float64(data) - MIN) / (MAX - MIN), decimals=4)

class FCNetwork(torch.nn.Module):
    def __init__(self, layers=[50, 20]):
        super(FCNetwork, self).__init__()
        dim_input  = 7   # 5 state features + 2 preference weights
        dim_output = 32  # 16 actions × 2 objectives
        net_layers = []
        dim = dim_input
        for layer_size in layers:
            net_layers.append(torch.nn.Linear(dim, layer_size))
            net_layers.append(torch.nn.ReLU())
            dim = layer_size
        net_layers.append(torch.nn.Linear(dim, dim_output))
        self.network = torch.nn.Sequential(*net_layers)

    def forward(self, states):
        states_tensor = torch.tensor(states, dtype=torch.float32)
        return self.network(states_tensor)


In [9]:
def get_data_dir(subfolder):
    current_dir = os.getcwd()
    print(current_dir)
    return os.path.join(current_dir, "experiment_data", f"{subfolder}")

DATA_DIR = get_data_dir("training_data")

csv_file_path = f'{DATA_DIR}/training_dataset.csv'


/home/cc/summer2024/main_codes


In [10]:
model = FCNetwork(layers=[20, 20])
policy_name = "trained_network_weights_20260331_150342_all_preference_model_based_0.01_0.001.pth"
policy_file = os.path.join("/home/cc/summer2024/main_codes/trained_models", policy_name)
model.load_state_dict(torch.load(policy_file))
model.eval()
print(f"Loaded: {policy_name}")


Loaded: trained_network_weights_20260331_150342_all_preference_model_based_0.01_0.001.pth


In [11]:
# import pickle
# import os

# # Define the path to the pickle file
# DATA_DIR = get_data_dir("training_data")
# pkl_file_path = os.path.join(DATA_DIR, 'normalization_bounds.pkl')

# # Load the pickle file
# with open(pkl_file_path, 'rb') as f:
#     bounds = pickle.load(f)

# # Extract the variables
# MAX_PROGRESS = bounds['MAX_PROGRESS']
# MIN_PROGRESS = bounds['MIN_PROGRESS']
# MAX_MPOWER = bounds['MAX_MPOWER']
# MIN_MPOWER = bounds['MIN_MPOWER']
# GLOBAL_MIN_PROGRESS = bounds['GLOBAL_MIN_PROGRESS']
# GLOBAL_MAX_PROGRESS = bounds['GLOBAL_MAX_PROGRESS']
# GLOBAL_MIN_MPOWER = bounds['GLOBAL_MIN_MPOWER']
# GLOBAL_MAX_MPOWER = bounds['GLOBAL_MAX_MPOWER']

# # Print to verify
# print("Loaded normalization bounds:")
# print(f"MAX_PROGRESS: {MAX_PROGRESS}")
# print(f"MIN_PROGRESS: {MIN_PROGRESS}")
# print(f"MAX_MPOWER: {MAX_MPOWER}")
# print(f"MIN_MPOWER: {MIN_MPOWER}")
# print(f"GLOBAL_MIN_PROGRESS: {GLOBAL_MIN_PROGRESS}")
# print(f"GLOBAL_MAX_PROGRESS: {GLOBAL_MAX_PROGRESS}")
# print(f"GLOBAL_MIN_MPOWER: {GLOBAL_MIN_MPOWER}")
# print(f"GLOBAL_MAX_MPOWER: {GLOBAL_MAX_MPOWER}")

# data = pd.read_csv(csv_file_path)

In [12]:
data = pd.read_csv(csv_file_path)
df = pd.DataFrame(data)

ACTIONS = [78.0, 83.0, 89.0, 95.0, 101.0, 107.0, 112.0, 118.0,
           124.0, 130.0, 136.0, 141.0, 147.0, 153.0, 159.0, 165.0]

# Filter out zero-state rows (uninformative initial states)
state_cols = ['Progress', 'Power', 'TOT_INS_PER_CYC', 'L3_TCM_PER_TCA', 'TOT_STL_PER_CYC']
df = df[~(df[state_cols] == 0).all(axis=1)].reset_index(drop=True)
print(f"Evaluating on {len(df)} rows (zero-state rows removed)")

# Compute per-app MAX_PROGRESS from the dataset (matches training)
MAX_PROGRESS = df.groupby('App')['Next_Progress'].max().to_dict()

application = ""  # filter by app name substring, or "" for all

for i in range(len(df)):
    if application in df.iloc[i]['App']:
        app        = df.iloc[i]['App']
        state_raw  = np.array(df.iloc[i][1:6], dtype=np.float32)  # (5,)

        # Build 2D preference vector matching training (get_tensors logic)
        prog_pref  = 0.1
        preference = np.array([1.0 - prog_pref, prog_pref], dtype=np.float32)  # (2,)

        # Concatenate state + preference → 7-dim input
        s_vecs   = np.concatenate([state_raw, preference], axis=0)  # (7,)
        s_vecs_t = torch.from_numpy(s_vecs).unsqueeze(0).float()    # (1, 7)

        # Forward pass → (1, 32) → (1, 16, 2)
        q_out    = model(s_vecs_t)
        act_vec  = q_out.view(1, 16, 2)          # (B=1, A=16, obj=2)

        # Scalarize using cosine similarity × dot product (matches training)
        pref_t        = torch.from_numpy(preference).unsqueeze(0)      # (1, 2)
        pref_flat     = pref_t.repeat_interleave(16, dim=0)            # (16, 2)
        q_flat        = act_vec.squeeze(0)                             # (16, 2)
        cos           = torch.clamp(
                            F.cosine_similarity(pref_flat, q_flat, dim=1), 0.0, 0.9999
                        )                                              # (16,)
        dot           = (pref_flat * q_flat).sum(dim=1)                # (16,)
        score         = (cos * dot).detach().numpy()                   # (16,)

        argmax = int(np.argmax(score))
        print(f"{i+1} ..... {app} ..... suggested PCAP: {ACTIONS[argmax]} W")


Evaluating on 316 rows (zero-state rows removed)
1 ..... ones-stream-copy ..... suggested PCAP: 89.0 W
2 ..... ones-stream-copy ..... suggested PCAP: 89.0 W
3 ..... ones-stream-copy ..... suggested PCAP: 89.0 W
4 ..... ones-stream-copy ..... suggested PCAP: 89.0 W
5 ..... ones-stream-copy ..... suggested PCAP: 89.0 W
6 ..... ones-stream-copy ..... suggested PCAP: 89.0 W
7 ..... ones-stream-copy ..... suggested PCAP: 89.0 W
8 ..... ones-stream-copy ..... suggested PCAP: 89.0 W
9 ..... ones-stream-copy ..... suggested PCAP: 89.0 W
10 ..... ones-stream-copy ..... suggested PCAP: 89.0 W
11 ..... ones-stream-copy ..... suggested PCAP: 89.0 W
12 ..... ones-stream-copy ..... suggested PCAP: 89.0 W
13 ..... ones-stream-copy ..... suggested PCAP: 89.0 W
14 ..... ones-stream-copy ..... suggested PCAP: 89.0 W
15 ..... ones-stream-copy ..... suggested PCAP: 89.0 W
16 ..... ones-stream-copy ..... suggested PCAP: 89.0 W
17 ..... ones-stream-copy ..... suggested PCAP: 89.0 W
18 ..... ones-stream-copy

/tmp/ipykernel_333812/3694844986.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  states_tensor = torch.tensor(states, dtype=torch.float32)


263 ..... ones-stream-triad ..... suggested PCAP: 89.0 W
264 ..... ones-stream-triad ..... suggested PCAP: 89.0 W
265 ..... ones-stream-triad ..... suggested PCAP: 89.0 W
266 ..... ones-stream-triad ..... suggested PCAP: 89.0 W
267 ..... ones-stream-triad ..... suggested PCAP: 89.0 W
268 ..... ones-stream-triad ..... suggested PCAP: 89.0 W
269 ..... ones-stream-triad ..... suggested PCAP: 89.0 W
270 ..... ones-stream-triad ..... suggested PCAP: 89.0 W
271 ..... ones-stream-triad ..... suggested PCAP: 89.0 W
272 ..... ones-stream-triad ..... suggested PCAP: 89.0 W
273 ..... ones-stream-triad ..... suggested PCAP: 89.0 W
274 ..... ones-stream-triad ..... suggested PCAP: 89.0 W
275 ..... ones-stream-triad ..... suggested PCAP: 89.0 W
276 ..... ones-stream-triad ..... suggested PCAP: 89.0 W
277 ..... ones-stream-triad ..... suggested PCAP: 89.0 W
278 ..... ones-stream-triad ..... suggested PCAP: 89.0 W
279 ..... ones-stream-triad ..... suggested PCAP: 89.0 W
280 ..... ones-stream-triad ...